# Wine Quality Prediction using K-Nearest Neighbors (KNN)

## **IT25101499**
## **Madubasha W M N**

# 1.Importing the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score)
from sklearn.decomposition import PCA

%matplotlib inline
sns.set(style="whitegrid")

## 2.Load the Dataset
Load the team's shared `processed_wine_quality.csv` (red + white wine combined, already standardized, with PCA components and an encoded `wine_type`). A readable `wine_type` label is derived from `wine_type_encoded` for the EDA plots.

In [ ]:
import os, json

# Team repo file - try the notebook folder, the parent folder, then data/
CANDIDATE_PATHS = ["processed_wine_quality.csv",
                   "../processed_wine_quality.csv",
                   "data/processed_wine_quality.csv"]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("processed_wine_quality.csv not found. Looked in: " + ", ".join(CANDIDATE_PATHS))
df = pd.read_csv(DATA_PATH)

# Preprocessing varieties used for modelling
FULL_FEATURES = ['fixed acidity_scaled', 'volatile acidity_scaled', 'citric acid_scaled',
                 'residual sugar_scaled', 'chlorides_scaled', 'free sulfur dioxide_scaled',
                 'total sulfur dioxide_scaled', 'density_scaled', 'pH_scaled',
                 'sulphates_scaled', 'alcohol_scaled', 'wine_type_encoded']   # Variety 1
PCA_FEATURES = ['PC1', 'PC2']                                                  # Variety 2

missing_cols = [c for c in FULL_FEATURES + PCA_FEATURES + ["quality"] if c not in df.columns]
if missing_cols:
    raise KeyError("Columns missing from " + DATA_PATH + ": " + str(missing_cols))

# Readable label for EDA plots (lower code = red, higher code = white)
df["wine_type"] = np.where(df["wine_type_encoded"] > df["wine_type_encoded"].min(), "white", "red")

print("Loaded:", DATA_PATH)
print("Combined shape:", df.shape)
print("Features:", list(df.columns))
df.head()

## 3.Exploratory Data Analysis - EDA
### shape, dtypes, head/tail, describe, missing, duplicates

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
print(df.describe())
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Quality value counts:\n", df["quality"].value_counts().sort_index())
print("Wine type counts:\n", df["wine_type"].value_counts())

# 4.Visual EDA: class balance, quality distribution

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="quality", data=df, ax=axes[0])
axes[0].set_title("Quality score distribution (multi-class)")
sns.countplot(x="wine_type", data=df, ax=axes[1])
axes[1].set_title("Red vs White counts")
plt.tight_layout(); plt.show()



# Correlation
Correlation heatmap (numeric cols only)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Correlation heatmap (all numeric features + quality)")
plt.show()

# Boxplots
Boxplots for outlier check

In [ ]:
num_cols = [c for c in FULL_FEATURES if c != "wine_type_encoded"]   # the 11 scaled numeric features
df[num_cols].plot(kind="box", subplots=True, layout=(3, 4), figsize=(14, 8), sharey=False)
plt.tight_layout(); plt.show()

# 5.Preprocessing
Create binary target `good_quality` and build two preprocessing varieties from `processed_wine_quality.csv`: **Variety 1** = 12 standardized features, **Variety 2** = PCA features (PC1, PC2). Stratified 80/20 split.

## Binary target
quality >= 7 is 'good' (handles class order + imbalance noted in winequality.names)

In [ ]:
df["good_quality"] = (df["quality"] >= 7).astype(int)
print(df["good_quality"].value_counts())
print(df["good_quality"].value_counts(normalize=True))

sns.countplot(x="good_quality", data=df)
plt.title("Binary target: 0 = bad (quality<7), 1 = good (quality>=7)")
plt.show()

## Features / label 
Raw quality is excluded from the inputs (target leakage).
##### Variety 1 = Full Features (12 standardized, incl. wine_type_encoded) | Variety 2 = PCA Features (PC1, PC2)

In [ ]:
X_full = df[FULL_FEATURES]     # Variety 1
X_pca = df[PCA_FEATURES]       # Variety 2
y = df["good_quality"]
print("Variety 1 (Full) X shape:", X_full.shape, "| Variety 2 (PCA) X shape:", X_pca.shape, "| y shape:", y.shape)
print(list(X_full.columns))
print(list(X_pca.columns))

# 6.Train/test split
Features in `processed_wine_quality.csv` are already standardized, so no scaler is fitted here. Both varieties share the same stratified 80/20 split.

In [ ]:
X_train_full, X_test_full, X_train_pca, X_test_pca, y_train, y_test = train_test_split(
    X_full, X_pca, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train_full.shape[0]}, Test: {X_test_full.shape[0]}")

# Variety 1 arrays under the names the baseline / PCA-plot cells below use
X_train_scaled = X_train_full.values
X_test_scaled = X_test_full.values

### verify features are standardized (mean ~0, std ~1)

In [ ]:
print(pd.DataFrame(X_train_scaled, columns=FULL_FEATURES).head())
print(pd.DataFrame(X_train_scaled, columns=FULL_FEATURES).describe().round(2))

## 7.Train KNN

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)
print("KNN(k=5) trained on", X_train_scaled.shape)

# 8.Evaluate
### BASELINE model: default KNN k=5 - single solid accuracy to submit

In [ ]:
y_pred_knn = model_knn.predict(X_test_scaled)
y_proba_knn = model_knn.predict_proba(X_test_scaled)[:, 1]

BASELINE_ACC = accuracy_score(y_test, y_pred_knn)
print("BASELINE ACCURACY k=5: %.4f" % BASELINE_ACC)
print("Precision:", round(precision_score(y_test, y_pred_knn), 4))
print("Recall   :", round(recall_score(y_test, y_pred_knn), 4))
print("F1-score :", round(f1_score(y_test, y_pred_knn), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, y_proba_knn), 4))
print("\n" + classification_report(y_test, y_pred_knn, target_names=["bad", "good"]))

cm = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["pred bad", "pred good"],
            yticklabels=["true bad", "true good"])
plt.title("Confusion Matrix - BASELINE KNN k=5")
plt.show()

# 9.Visualize Decision Boundary via PCA
KNN has 12 features so we compress to 2D with PCA purely for visualization, then retrain a 2D KNN.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)
print("Explained variance:", pca.explained_variance_ratio_.round(3))

viz_knn = KNeighborsClassifier(n_neighbors=5)
viz_knn.fit(X_train_2d, y_train)

xx, yy = np.meshgrid(np.linspace(X_train_2d[:, 0].min()-1, X_train_2d[:, 0].max()+1, 200),
                     np.linspace(X_train_2d[:, 1].min()-1, X_train_2d[:, 1].max()+1, 200))
Z = viz_knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
plt.scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", s=20)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("KNN (k=5) decision boundary on PCA-2D + test points")
plt.show()

# 10.Hyperparameter Tuning (GridSearchCV on both varieties)
Replaces the manual K sweep. 5-fold Stratified CV, `scoring='f1_weighted'`, searching `n_neighbors`, `weights` and `metric` separately on **Variety 1 (Full)** and **Variety 2 (PCA)**.

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

varieties = {
    "Variety 1 - Full Features": (X_train_full, X_test_full),
    "Variety 2 - PCA Features":  (X_train_pca, X_test_pca),
}

grids = {}
for name, (Xtr, Xte) in varieties.items():
    g = GridSearchCV(KNeighborsClassifier(), param_grid, cv=cv, scoring="f1_weighted", n_jobs=-1)
    g.fit(Xtr, y_train)
    grids[name] = g
    print(name)
    print("  Best CV weighted F1: %.4f" % g.best_score_)
    print("  Best params        :", g.best_params_)

# K vs CV weighted F1 (best setting for each k), both varieties
fig, ax = plt.subplots(figsize=(7, 4))
for name, g in grids.items():
    r = pd.DataFrame(g.cv_results_)
    r["k"] = r["param_n_neighbors"].astype(int)
    r.groupby("k")["mean_test_score"].max().plot(marker="o", ax=ax, label=name)
ax.set_xlabel("n_neighbors"); ax.set_ylabel("CV weighted F1")
ax.set_title("K vs CV weighted F1 (GridSearchCV)")
ax.legend(); ax.grid(True); plt.show()

# 11.Evaluate the winning model
The variety with the higher CV weighted F1 wins (chosen on CV only, never on the test set). The winner is then scored once on the held-out test set. Its accuracy is compared with the BASELINE.

In [ ]:
# Both varieties on the test set (for the comparison table)
rows = []
for name, (Xtr, Xte) in varieties.items():
    p = grids[name].best_estimator_.predict(Xte)
    rows.append({"variety": name, "cv_weighted_f1": grids[name].best_score_,
                 "test_accuracy": accuracy_score(y_test, p),
                 "test_weighted_f1": f1_score(y_test, p, average="weighted"),
                 "best_params": str(grids[name].best_params_)})
variety_table = pd.DataFrame(rows)
print(variety_table.to_string(index=False))

# Winner = highest CV weighted F1
winner_name = max(grids, key=lambda n: grids[n].best_score_)
grid = grids[winner_name]
X_test_win = varieties[winner_name][1]
print("\nWINNING PREPROCESSING:", winner_name)
print("Best params:", grid.best_params_)
print("Best CV weighted F1: %.4f" % grid.best_score_)

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_win)
y_proba_best = best_model.predict_proba(X_test_win)[:, 1]

TUNED_ACC = accuracy_score(y_test, y_pred_best)
print("\nFINAL TEST METRICS (winning model)")
print("Accuracy      : %.4f" % TUNED_ACC)
print("Precision     : %.4f (weighted)" % precision_score(y_test, y_pred_best, average="weighted"))
print("Recall        : %.4f (weighted)" % recall_score(y_test, y_pred_best, average="weighted"))
print("Weighted F1   : %.4f" % f1_score(y_test, y_pred_best, average="weighted"))
print("Macro F1      : %.4f" % f1_score(y_test, y_pred_best, average="macro"))
print("ROC AUC       : %.4f" % roc_auc_score(y_test, y_proba_best))
print("BASELINE TEST ACCURACY (k=5, full features): %.4f" % BASELINE_ACC)
print("IMPROVEMENT: %+.4f" % (TUNED_ACC - BASELINE_ACC))
print("\n" + classification_report(y_test, y_pred_best, target_names=["bad", "good"]))

# 12.Accuracy

In [ ]:
summary = pd.DataFrame({"model": ["BASELINE k=5", str(grid.best_params_)], "accuracy": [BASELINE_ACC, TUNED_ACC]})
print(summary)

### Barchart

In [ ]:
sns.barplot(x="model", y="accuracy", data=summary)
plt.ylim(max(0, min(BASELINE_ACC, TUNED_ACC)-0.05), 1.0)
plt.title("Baseline vs Tuned KNN accuracy")
for i, v in enumerate(summary["accuracy"]):
    plt.text(i, v+0.005, "%.4f" % v, ha="center")
plt.show()


### Comparison Table

In [ ]:
# Output folder: reuse an existing results folder, otherwise create one next to the data
RESULTS_DIR = next((d for d in ["../results", "results"] if os.path.isdir(d)),
                   "../results" if DATA_PATH.startswith("..") else "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt="d", cmap="Greens",
            xticklabels=["pred bad", "pred good"], yticklabels=["true bad", "true good"])
plt.title("Confusion Matrix - TUNED KNN (" + winner_name + ")")
plt.tight_layout()
for fname in ["IT25101499_cm.png", "knn_cm.png"]:
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion matrix to", RESULTS_DIR)

### ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_val = roc_auc_score(y_test, y_proba_best)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label="KNN (AUC = %.4f)" % auc_val)
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - TUNED KNN (" + winner_name + ")")
plt.legend(loc="lower right")
plt.tight_layout()
for fname in ["IT25101499_roc.png", "knn_roc.png"]:
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
plt.show()
print("Saved ROC curve to", RESULTS_DIR)

# 13.Export metrics for the group comparison
Writes `IT25101499_metrics.json` (same keys as the other team notebooks) so `Group.ipynb` can load it.

In [ ]:
metrics_out = {
    "student_id": "IT25101499",
    "student_name": "Madubasha W M N",
    "model": "KNN",
    "best_params": {k: (int(v) if isinstance(v, (int, np.integer)) else v) for k, v in grid.best_params_.items()},
    "cv_weighted_f1": float(grid.best_score_),
    "accuracy": float(accuracy_score(y_test, y_pred_best)),
    "precision": float(precision_score(y_test, y_pred_best, average="weighted")),
    "recall": float(recall_score(y_test, y_pred_best, average="weighted")),
    "weighted_f1": float(f1_score(y_test, y_pred_best, average="weighted")),
    "macro_f1": float(f1_score(y_test, y_pred_best, average="macro")),
    "roc_auc": float(roc_auc_score(y_test, y_proba_best)),
    "confusion_matrix": confusion_matrix(y_test, y_pred_best).tolist(),
    "roc_curve": {"fpr": [float(v) for v in fpr], "tpr": [float(v) for v in tpr]},
}

json_path = os.path.join(RESULTS_DIR, "IT25101499_metrics.json")
with open(json_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Saved", json_path)
print({k: v for k, v in metrics_out.items() if k not in ("confusion_matrix", "roc_curve")})

## Conclusion
- Baseline: default KNN (k=5, Euclidean, uniform weights) on the 12 standardized features - single reference accuracy.
- Tuning: 5-fold Stratified `GridSearchCV` (`f1_weighted`) over `n_neighbors`, `weights` and `metric`, run on Variety 1 (Full Features) and Variety 2 (PCA PC1/PC2). The variety with the higher CV weighted F1 is the winner (see the comparison table in section 11).
- Final test metrics (Accuracy, Precision, Recall, Weighted F1, Macro F1, ROC-AUC), the confusion matrix, the ROC curve and `IT25101499_metrics.json` are all produced from the winning model.
- Why it can boost: distance weighting and Manhattan distance can help on imbalanced, skewed wine features; larger K smooths noisy boundaries. A 2-component PCA keeps less information than the full 12 features, which the comparison table makes visible.